In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kruskal, chi2_contingency, linregress
import seaborn as sns
from lifelines import KaplanMeierFitter, CoxPHFitter
from sklearn.utils.class_weight import compute_class_weight
import statsmodels
from statsmodels.stats.multitest import multipletests
from matplotlib import rcParams

In [ ]:
RANDOM_STATE = 42
n_clusters = 2

In [ ]:
sns.set_theme(style='ticks')
rcParams.update({
    'font.size': 11,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica'],
    'axes.labelsize': 11,
    'axes.titlesize': 11,
    'axes.edgecolor': 'black',
    'axes.linewidth': 0.8,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
    'xtick.major.size': 3,
    'ytick.major.size': 3,
    'legend.fontsize': 11,
    'legend.frameon': False,
    'savefig.format': 'svg',
    'savefig.dpi': 300,  # Still useful for rasterized elements
    'figure.dpi': 100,
    'figure.figsize': (3.5, 2.5),  # Approx. half-column width
    'figure.constrained_layout.use': True,
    'svg.fonttype': 'none',  # Keep text as editable text (not paths)
    'axes.spines.top': False,
    'axes.spines.right': False,
})

In [ ]:
import warnings
warnings.filterwarnings("ignore")
final_clusters = pd.read_csv('../data/patient_clusters.csv', index_col=0).T.to_dict('list')
final_clusters = {k: [x for x in v if pd.notna(x)] for k, v in final_clusters.items()}
clinical_data_file = pd.read_csv('../data/TCGA/omics_data/raw/cancer_data_PAAD_clinical_data.tsv', sep='\t', header=0)

In [ ]:
clinical_data = clinical_data_file[['Patient ID', 'Mutation Count', 'Fraction Genome Altered', 'Diagnosis Age', 'Sex', 'Race Category', 
                                    'Adjuvant Postoperative Targeted Therapy Administered Indicator', 'Alcohol History Documented', 'Tumor resected max dimension',
                                    'American Joint Committee on Cancer Metastasis Stage Code', 'American Joint Committee on Cancer Tumor Stage Code',
                                    'Chronic Pancreatitis Personal Medical History Indicator', 'Did patient start adjuvant postoperative radiotherapy?', 
                                    'Disease Free Status', 'Family History of Cancer', 'Neoplasm Disease Lymph Node Stage American Joint Committee on Cancer Code', 
                                    'Neoplasm Disease Stage American Joint Committee on Cancer Code', 'Neoplasm Histologic Grade', 'TMB (nonsynonymous)',
                                    'New Neoplasm Event Post Initial Therapy Indicator', 'Overall Survival (Months)', 'Overall Survival Status', 'Disease Free (Months)', 
                                    'Participant Personal Medical History Diabetes Mellitus Ind-3', 'Patient Primary Tumor Site', 'Prior Cancer Diagnosis Occurence', 
                                    'Surgical Margin Resection Status', 'Patient Smoking History Category', 'Person Neoplasm Status', 'Primary Therapy Outcome Success Type']]
clinical_data['Overall Survival Status'] = clinical_data['Overall Survival Status'].str.split(':').str[0].astype(int)
clinical_data['Patient Smoking History Category'] = (clinical_data['Patient Smoking History Category']
                                                         .where(clinical_data['Patient Smoking History Category'].isna(), 
                                                                clinical_data['Patient Smoking History Category'].astype(float).astype(str)))
clinical_data.set_index('Patient ID', inplace=True)
clinical_data['Cluster'] = None
for cluster_num, patients in final_clusters.items():
    cluster_label = 1 if int(cluster_num.split('_')[-1]) == 1 else 2
    clinical_data.loc[clinical_data.index.isin(patients), 'Cluster'] = cluster_label
clinical_data.dropna(subset=['Cluster'], inplace=True)

# Clinical analysis

### Survival curves

In [ ]:
# Check if weights are needed
test_weights_survival = clinical_data[['Overall Survival Status', 'Cluster']]
test_weights_survival['Cluster'] = pd.to_numeric(test_weights_survival['Cluster'])
result_weights_survival = linregress(x=test_weights_survival['Overall Survival Status'].values, y=test_weights_survival['Cluster'].values)
print(f"p = {result_weights_survival.pvalue}")

Since the p-value is < 0.05, we can assume that censoring through time isn't random, hence weights are needed to calculate the overall p-value for the log-rank test.

In [ ]:
survival_df = clinical_data[['Overall Survival Status', 'Overall Survival (Months)', 'Cluster']]
survival_df['Weights'] = None
for i in np.arange(min(survival_df['Overall Survival (Months)']), max(survival_df['Overall Survival (Months)']), 6):
    interval_df = survival_df[(survival_df['Overall Survival (Months)'] >= i) & (survival_df['Overall Survival (Months)'] < i + 6)].copy()
    censored_data = interval_df['Overall Survival Status'].to_numpy()
    unique_classes = np.unique(censored_data)
    if censored_data.size != 0:
        class_weights = compute_class_weight(class_weight='balanced', classes=unique_classes, y=censored_data)
        weight_mapping = dict(zip(unique_classes, class_weights))
        interval_df['Weights'] = interval_df['Overall Survival Status'].map(weight_mapping)
        survival_df.update(interval_df[['Weights']])
survival_df['Weights'] = pd.to_numeric(survival_df['Weights'])
survival_df.to_csv("../results/cluster_analysis/survival_diseasefree_analysis/survival_final_clusters.csv")
survival_df.sort_values(by='Cluster', ascending=True, inplace=True)    # order to have analysis with respect to cluster 1

In [ ]:
colorblind_palette = ["#F8766D", "#00BFC4"]
plt.figure(figsize=(9, 4))
kmf_list = []
for cluster in survival_df['Cluster'].unique():
    cluster_data = survival_df[survival_df['Cluster'] == cluster]
    kmf = KaplanMeierFitter()
    kmf.fit(cluster_data['Overall Survival (Months)'], cluster_data['Overall Survival Status'],
            label=f'Cluster {int(cluster) +1}')
    kmf.plot_survival_function(show_censors=True, color=colorblind_palette[cluster])
    kmf_list.append(kmf)
plt.xlabel('Time (months)')
max_time = max(survival_df['Overall Survival (Months)'])
plt.xticks(list(np.arange(0, max_time+3, 6)))
plt.ylabel('Survival probability')

cph_data = survival_df.reset_index(drop=True)
cph = CoxPHFitter()
cph_df = cph.fit(cph_data, duration_col='Overall Survival (Months)', event_col='Overall Survival Status', weights_col='Weights').summary
print(f"HR = {cph_df['exp(coef)'].loc['Cluster']:.2f}, CI: ({cph_df['exp(coef) lower 95%'].loc['Cluster']:.2f}, {cph_df['exp(coef) upper 95%'].loc['Cluster']:.2f})")

plt.savefig('FIGURES/clinical_analysis/survival_final_clusters.svg', bbox_inches='tight')
plt.show()

In [ ]:
# Cox Proportional hazards, Fisher's test and RMST results at 6, 12, 36 and 60 months
months = [6, 12, 18, 24, 36, 60]
print('Hazard ratios')
for time in months: 
    print(f"{time} months:")
    data_surv = survival_df.copy()
    time_subset = survival_df['Overall Survival (Months)'] <= time
    data_surv.loc[~time_subset, 'Overall Survival Status'] = 0
    # Cox proportional hazards model
    cph_data_surv = data_surv.reset_index(drop=True)
    cph = CoxPHFitter()
    cph_surv = cph.fit(cph_data_surv, duration_col='Overall Survival (Months)', event_col='Overall Survival Status').summary
    print(f"Hazard Ratio = {cph_surv['exp(coef)'].loc['Cluster']:.2f}, CI: ({cph_surv['exp(coef) lower 95%'].loc['Cluster']:.2f}, {cph_surv['exp(coef) upper 95%'].loc['Cluster']:.2f}), p-val = {cph_surv['p'].loc['Cluster']:.2f}")
    # Test if censoring is random
    data_surv['Cluster'] = pd.to_numeric(data_surv['Cluster'])
    test_censoring = linregress(x=data_surv['Overall Survival Status'].values, y=data_surv['Cluster'].values)
    print(f"Random censoring: p = {test_censoring.pvalue}")

### Disease free curves

In [ ]:
test_weights_disease_free = clinical_data[['Disease Free Status', 'Cluster']]
test_weights_disease_free = test_weights_disease_free.dropna(how='any',axis=0)
test_weights_disease_free['Cluster'] = pd.to_numeric(test_weights_disease_free['Cluster'])
test_weights_disease_free['Disease Free Status'] = test_weights_disease_free['Disease Free Status'].str.split(':').str[0].astype(int)
result_weights_disease_free = linregress(x=test_weights_disease_free['Disease Free Status'].values, y=test_weights_disease_free['Cluster'].values)
print(f"p = {result_weights_disease_free.pvalue}")

Since the p-value is > 0.05, we have insufficient evidence to reject the null hypothesis, hence assume the censoring was random, and don't need weights for future calculations.

In [ ]:
disease_free_df = clinical_data[['Disease Free Status', 'Disease Free (Months)', 'Cluster']]
disease_free_df.sort_values(by='Cluster', ascending=True, inplace=True)    # order to have analysis with respect to cluster 1
disease_free_df = disease_free_df.dropna(how='any',axis=0)
disease_free_df['Disease Free Status'] = disease_free_df['Disease Free Status'].str.split(':').str[0].astype(int)
disease_free_df.to_csv("../results/cluster_analysis/survival_diseasefree_analysis/disease_free_final_clusters.csv")

In [ ]:
kmf_list = []
plt.figure(figsize=(9, 4))
for cluster in sorted(disease_free_df['Cluster'].unique()):
    cluster_data = disease_free_df[disease_free_df['Cluster'] == cluster]
    kmf = KaplanMeierFitter()
    kmf.fit(cluster_data['Disease Free (Months)'], cluster_data['Disease Free Status'],
            label=f'Cluster {int(cluster)}')
    kmf.plot_survival_function(show_censors=True, color=colorblind_palette[cluster-1])
    kmf_list.append(kmf)
max_time= max(disease_free_df['Disease Free (Months)'])
plt.xticks(list(np.arange(0, max_time+3, 6)))
plt.xlabel('Time (months)')
plt.ylabel('Disease free probability')

cph_data = disease_free_df.reset_index(drop=True)
cph = CoxPHFitter()
cph_df = cph.fit(cph_data, duration_col='Disease Free (Months)', event_col='Disease Free Status').summary
print(f"HR = {cph_df['exp(coef)'].loc['Cluster']:.2f}, CI: ({cph_df['exp(coef) lower 95%'].loc['Cluster']:.2f}, {cph_df['exp(coef) upper 95%'].loc['Cluster']:.2f})")

plt.savefig('FIGURES/clinical_analysis/diseasefree_final_clusters.svg', bbox_inches='tight')
plt.show()

In [ ]:
# Cox Proportional hazards results at 6, 12, 36 and 60 months
months = [6, 12, 18, 24, 36, 60]
for time in months: 
    print(f"{time} months:")
    data_diseasefree = disease_free_df.copy()
    time_subset = disease_free_df['Disease Free (Months)'] <= time
    data_diseasefree.loc[~time_subset, 'Disease Free Status'] = 0
    # Cox proportional hazards model
    cph_data_diseasefree = data_diseasefree.reset_index(drop=True)
    cph = CoxPHFitter()
    cph_diseasefree = cph.fit(cph_data_diseasefree, duration_col='Disease Free (Months)', event_col='Disease Free Status').summary
    print(f"Hazard Ratio = {cph_diseasefree['exp(coef)'].loc['Cluster']:.2f}, CI: ({cph_diseasefree['exp(coef) lower 95%'].loc['Cluster']:.2f}, {cph_diseasefree['exp(coef) upper 95%'].loc['Cluster']:.2f}), p-val = {cph_diseasefree['p'].loc['Cluster']:.2f}")
    # Test if censoring is random
    data_diseasefree['Cluster'] = pd.to_numeric(data_diseasefree['Cluster'])
    test_censoring = linregress(x=data_diseasefree['Disease Free Status'].values, y=data_diseasefree['Cluster'].values)
    print(f"Random censoring: p = {test_censoring.pvalue}")

### Enrichment of clinical labels

In [ ]:
# Enrichment of clinical labels
clinical_labels = clinical_data.copy()
clinical_enrichment = {}
for variable in clinical_labels.columns:
    if pd.api.types.is_numeric_dtype(clinical_labels[variable]):
        test_numerical = [
            clinical_labels[clinical_labels['Cluster'] == cluster][variable].dropna().to_numpy() 
            for cluster in clinical_labels['Cluster'].unique()]
        stat, p_value_kruskal = kruskal(*test_numerical)
        clinical_enrichment[variable] = p_value_kruskal
    else: 
        test_discrete = pd.crosstab(clinical_labels['Cluster'], clinical_labels[variable])
        chi2, p_value_chi2, dof, expected = chi2_contingency(test_discrete)
        clinical_enrichment[variable] = p_value_chi2
del (clinical_enrichment['Cluster'], clinical_enrichment['Overall Survival (Months)'], clinical_enrichment['Overall Survival Status'], 
     clinical_enrichment['Disease Free (Months)'], clinical_enrichment['Disease Free Status'])
clinical_enrichment_pvalues = pd.DataFrame.from_dict(clinical_enrichment, orient='index', columns=['Original p-value'])
reject, pvals_corr, asidack, abonf = statsmodels.stats.multitest.multipletests(pvals=clinical_enrichment_pvalues['Original p-value'], alpha=0.05, 
                                                                               method='fdr_bh', maxiter=1, is_sorted=False, returnsorted=False)
clinical_enrichment_pvalues['Adjusted p-value'] = pvals_corr
clinical_enrichment_pvalues['Significance'] = clinical_enrichment_pvalues['Adjusted p-value'].apply(lambda x: '*' if x < 0.05 else '')
clinical_enrichment_pvalues[clinical_enrichment_pvalues['Significance'] == '*']

In [ ]:
palette = ["#F8766D", "#00BFC4"]
fig, ax = plt.subplots(1, 3, figsize=(12, 3))
sns.boxplot(data=clinical_data, x='Cluster', y='Mutation Count', showmeans=True, palette=palette, ax=ax[0], meanprops={"markerfacecolor": "#800026","markeredgecolor": "#800026"})
ax[0].text(x=0.2, y=0.9, s=f"p = {clinical_enrichment_pvalues['Adjusted p-value'].loc['Mutation Count']:.3e}", ha='center', va='top', transform=ax[0].transAxes)

sns.boxplot(data=clinical_data, x='Cluster', y='Fraction Genome Altered', showmeans=True, palette=palette, ax=ax[1], meanprops={"markerfacecolor": "#800026","markeredgecolor": "#800026"})
ax[1].text(x=0.2, y=0.9, s=f"p = {clinical_enrichment_pvalues['Adjusted p-value'].loc['Fraction Genome Altered']:.3e}", ha='center', va='top', transform=ax[1].transAxes)
ax[1].set_ylabel('Fraction Genome Altered (%)')

sns.boxplot(data=clinical_data, x='Cluster', y='TMB (nonsynonymous)', showmeans=True, palette=palette, ax=ax[2], meanprops={"markerfacecolor": "#800026","markeredgecolor": "#800026"})
ax[2].text(x=0.2, y=0.9, s=f"p = {clinical_enrichment_pvalues['Adjusted p-value'].loc['TMB (nonsynonymous)']:.3e}", ha='center', va='top', transform=ax[2].transAxes)

plt.tight_layout(w_pad=4)
plt.show()

There seems to be an extreme outlier in mutation count, so let's repeat the test after removing the outlier patient.

In [ ]:
outlier = clinical_data[clinical_data['Mutation Count'] > 500].index    # isolate the outlier from the plots
clinical_data_no_outlier = clinical_data.drop(outlier)
clinical_enrichment_no_outlier = {}
for variable in clinical_data_no_outlier.columns:
    if pd.api.types.is_numeric_dtype(clinical_data_no_outlier[variable]):
        test_numerical = [
            clinical_data_no_outlier[clinical_data_no_outlier['Cluster'] == cluster][variable].dropna().to_numpy() 
            for cluster in clinical_data_no_outlier['Cluster'].unique()]
        stat, p_value_kruskal = kruskal(*test_numerical)
        clinical_enrichment_no_outlier[variable] = p_value_kruskal
    else: 
        test_discrete = pd.crosstab(clinical_data_no_outlier['Cluster'], clinical_data_no_outlier[variable])
        chi2, p_value_chi2, dof, expected = chi2_contingency(test_discrete)
        clinical_enrichment_no_outlier[variable] = p_value_chi2
del (clinical_enrichment_no_outlier['Cluster'], clinical_enrichment_no_outlier['Overall Survival (Months)'], clinical_enrichment_no_outlier['Overall Survival Status'], 
     clinical_enrichment_no_outlier['Disease Free Status'], clinical_enrichment_no_outlier['Disease Free (Months)'])
clinical_enrichment_pvalues_no_outlier = pd.DataFrame.from_dict(clinical_enrichment_no_outlier, orient='index', columns=['Original p-value'])
reject, pvals_corr, asidack, abonf = statsmodels.stats.multitest.multipletests(pvals=clinical_enrichment_pvalues_no_outlier['Original p-value'], alpha=0.05, 
                                                                               method='fdr_bh', maxiter=1, is_sorted=False, returnsorted=False)
clinical_enrichment_pvalues_no_outlier['Adjusted p-value'] = pvals_corr
clinical_enrichment_pvalues_no_outlier['Significance'] = clinical_enrichment_pvalues_no_outlier['Adjusted p-value'].apply(lambda x: '*' if x < 0.05 else '')
clinical_enrichment_pvalues_no_outlier[clinical_enrichment_pvalues_no_outlier['Significance'] == '*']

In [ ]:
variables_df = pd.DataFrame(columns=['Variable name', 'Type of variable', 'Possible values', 'Statistical test', 'P-value (adjusted)'])
variables_df[['Variable name', 'P-value (adjusted)']] = clinical_enrichment_pvalues.reset_index().iloc[:, [0, 2]]
variables_df['P-value (adjusted)'] = variables_df['P-value (adjusted)'].map(lambda x: float(f"{x:.3g}"))
for variable in variables_df['Variable name']:
    variables_df.loc[variables_df['Variable name'] == variable, 'Type of variable'] = clinical_data[variable].dtype
    variables_df.loc[(variables_df['Variable name'] == variable) & (variables_df['Type of variable'].isin([np.dtype('float64'), np.dtype('int64')])), 
                     'Statistical test'] = 'Kruskal Wallis'
    variables_df.loc[(variables_df['Variable name'] == variable) & (variables_df['Type of variable'] == np.object_), 
                     'Statistical test'] = 'Chi-square'
    variables_df.loc[(variables_df['Variable name'] == variable) & (variables_df['Type of variable'] == np.object_), 
                     'Possible values'] = f'({clinical_data[variable].unique()})'
    variables_df.loc[(variables_df['Variable name'] == variable) & (variables_df['Type of variable'].isin([np.dtype('float64'), np.dtype('int64')])),
                     'Possible values'] = f'({pd.to_numeric(clinical_data[variable], errors="coerce").min()} - {pd.to_numeric(clinical_data[variable], errors="coerce").max()})'
#variables_df

In [ ]:
from statannotations.Annotator import Annotator
pairs = [(1, 2)]
fig, ax = plt.subplots(3, 1, figsize=(4, 16))
sns.boxplot(data=clinical_data_no_outlier, x='Cluster', y='Mutation Count', showmeans=True, palette=palette, ax=ax[0], meanprops={"markerfacecolor": "#800026","markeredgecolor": "#800026"})
p_value_mutcount = clinical_enrichment_pvalues['Adjusted p-value'].loc['Mutation Count']
annotator_mutcount = Annotator(ax[0], pairs, data=clinical_data_no_outlier, x='Cluster', y='Mutation Count')
annotator_mutcount.configure(test=None, text_format="simple", loc="inside", verbose=2)
annotator_mutcount.set_custom_annotations([f"p = {p_value_mutcount:.2e}"])
annotator_mutcount.annotate()
ax[0].set_ylabel('Mutation Count')

sns.boxplot(data=clinical_data_no_outlier, x='Cluster', y='Fraction Genome Altered', showmeans=True, palette=palette, ax=ax[1], meanprops={"markerfacecolor": "#800026","markeredgecolor": "#800026"})
p_value_fga = clinical_enrichment_pvalues['Adjusted p-value'].loc['Fraction Genome Altered']
annotator_fga = Annotator(ax[1], pairs, data=clinical_data_no_outlier, x='Cluster', y='Fraction Genome Altered')
annotator_fga.configure(test=None, text_format="simple", loc="inside", verbose=2)
annotator_fga.set_custom_annotations([f"p = {p_value_fga:.2e}"])
annotator_fga.annotate()
ax[1].set_ylabel('Fraction Genome Altered')

sns.boxplot(data=clinical_data_no_outlier, x='Cluster', y='TMB (nonsynonymous)', showmeans=True, palette=palette, ax=ax[2], meanprops={"markerfacecolor": "#800026","markeredgecolor": "#800026"})
p_value_tmb = clinical_enrichment_pvalues['Adjusted p-value'].loc['TMB (nonsynonymous)']
annotator_tmb = Annotator(ax[2], pairs, data=clinical_data_no_outlier, x='Cluster', y='TMB (nonsynonymous)')
annotator_tmb.configure(test=None, text_format="simple", loc="inside", verbose=2)
annotator_tmb.set_custom_annotations([f"p = {p_value_tmb:.2e}"])
annotator_tmb.annotate()
ax[2].set_ylabel('Tumor Mutational Burden (nonsynonymous)')

plt.tight_layout(h_pad=2)
plt.savefig('FIGURES/clinical_analysis/enriched_labels_boxplots.svg', bbox_inches='tight')
plt.show()

The plots show the clinical variables with statistically significant differences between groups. The data used to plot the figures has the outlier removed to better visualise the differences, however the p-value is that of the statistical test including the outlier.